In [ ]:
# Installing necessary libraries
!pip install requests

# Importing necessary libraries
from google.colab import drive
from os import path, mkdir, getcwd, chdir, getenv
from sys import exit, stdout, stderr
from typing import Dict, List, Set
from requests import get, post
from concurrent.futures import ThreadPoolExecutor
from hashlib import sha256
from shutil import move
from time import perf_counter
import re

# Mount Google Drive
drive.mount('/content/gdrive')

# Change the working directory to the Google Drive folder
chdir('/content/gdrive/My Drive')

NEW_LINE: str = "\n"

def die(_str: str) -> None:
    stderr.write(_str + NEW_LINE)
    stderr.flush()
    exit(-1)

def _print(_str: str) -> None:
    stdout.write(_str)
    stdout.flush()

# increase max_workers for parallel downloads
# defaults to 5 download at time
class Main:
    def __init__(self, url: str, password: str | None = None, max_workers: int = 5) -> None:
        try:
            # Extract content ID from URL using regex
            if match := re.search(r'gofile\.io/d/([a-zA-Z0-9]+)', url):
                self._id = match.group(1)
            else:
                die(f"Could not extract content ID from the URL: {url}")
        except Exception as e:
            die(f"Error processing URL {url}: {str(e)}")

        self._downloaddir: str | None = getenv("GF_DOWNLOADDIR")

        if self._downloaddir and path.exists(self._downloaddir):
            chdir(self._downloaddir)

        self._token: str = self._getToken()
        self._password: str | None = sha256(password.encode()).hexdigest() if password else None
        self._max_workers: int = max_workers

        # list of dictionaries format files and its respective path, filename and link
        self._files_link_list: List[Dict] = []
        # List to store all available files
        self._available_files: List[Dict] = []

        # Get content info and process files
        content_data = self._getContentInfo(self._id, self._token, self._password)
        if content_data["type"] == "folder":
            self._root_dir = path.join(getcwd(), content_data["name"])
            self._createDir(content_data["name"])
            chdir(content_data["name"])
        else:
            self._root_dir = getcwd()

        # First collect all available files
        self._collectAvailableFiles(content_data)

        # Display files and get user selection
        selected_files = self._selectFiles()

        # Process only selected files
        self._processSelectedFiles(selected_files)
        self._threadedDownloads()

    def _getContentInfo(self, content_id: str, token: str, password: str | None = None) -> Dict:
        """Get content information from Gofile API"""
        url = f"https://api.gofile.io/contents/{content_id}?wt=4fd6sg89d7s6&contentFilter=&page=1&pageSize=1000&sortField=name&sortDirection=1"
        if password:
            url = url + f"&password={password}"

        headers: Dict = {
            "User-Agent": getenv("GF_USERAGENT") if getenv("GF_USERAGENT") else "Mozilla/5.0",
            "Accept-Encoding": "gzip, deflate, br",
            "Accept": "*/*",
            "Connection": "keep-alive",
            "Authorization": "Bearer" + " " + token,
        }

        response: Dict = get(url, headers=headers).json()
        if response["status"] != "ok":
            die(f"Failed to get content info from {url}")

        data: Dict = response["data"]
        if not data.get("canAccess", False):
            die("Cannot access the content. It might be private or password protected.")

        return data

    def _collectAvailableFiles(self, content_data: Dict, current_path: str = "") -> None:
        """Collect all available files in the content"""
        if content_data["type"] == "folder":
            folder_path = path.join(current_path, content_data["name"])
            for child in content_data["children"].values():
                if child["type"] == "folder":
                    subfolder_data = self._getContentInfo(child["code"], self._token, self._password)
                    self._collectAvailableFiles(subfolder_data, folder_path)
                else:
                    self._available_files.append({
                        "path": folder_path,
                        "name": child["name"],
                        "size": child["size"],
                        "link": child["link"]
                    })
        else:
            self._available_files.append({
                "path": current_path,
                "name": content_data["name"],
                "size": content_data.get("size", 0),
                "link": content_data["link"]
            })

    def _selectFiles(self) -> Set[int]:
        """Display available files and let user select which ones to download"""
        if not self._available_files:
            die("No files found in the content")

        _print("\nAvailable files:\n")
        for i, file in enumerate(self._available_files, 1):
            size_mb = file["size"] / (1024 * 1024)
            _print(f"{i}. {path.join(file['path'], file['name'])} ({size_mb:.2f} MB)\n")

        while True:
            _print("\nEnter the numbers of files you want to download (comma-separated, or 'all' for all files): ")
            selection = input().strip().lower()

            if selection == 'all':
                return set(range(len(self._available_files)))

            try:
                selected = set()
                for num in selection.split(','):
                    num = int(num.strip()) - 1
                    if 0 <= num < len(self._available_files):
                        selected.add(num)
                    else:
                        raise ValueError()
                if selected:
                    return selected
                _print("Please select at least one file\n")
            except ValueError:
                _print("Invalid input. Please enter comma-separated numbers or 'all'\n")

    def _processSelectedFiles(self, selected_indices: Set[int]) -> None:
        """Process only the selected files"""
        for idx in selected_indices:
            file = self._available_files[idx]
            full_path = path.join(file["path"], "")
            if full_path:
                self._createDir(full_path)
            self._cacheLink(path.join(self._root_dir, full_path), file["name"], file["link"])

    def _threadedDownloads(self) -> None:
        chdir(self._root_dir)
        with ThreadPoolExecutor(max_workers=self._max_workers) as executor:
            for item in self._files_link_list:
                executor.submit(self._downloadContent, item, self._token, 16384)

    def _createDir(self, dirname: str) -> None:
        if not dirname:
            return

        current_dir: str = getcwd()
        filepath: str = path.join(current_dir, dirname)
        try:
            mkdir(path.join(filepath))
        except FileExistsError:
            pass
        except Exception as e:
            if not path.exists(filepath):
                # Create intermediate directories if they don't exist
                path_parts = path.normpath(dirname).split(path.sep)
                current = current_dir
                for part in path_parts:
                    if part:
                        current = path.join(current, part)
                        if not path.exists(current):
                            mkdir(current)

    @staticmethod
    def _getToken() -> str:
        headers: Dict = {
            "User-Agent": getenv("GF_USERAGENT") if getenv("GF_USERAGENT") else "Mozilla/5.0",
            "Accept-Encoding": "gzip, deflate, br",
            "Accept": "*/*",
            "Connection": "keep-alive",
        }
        create_account_response: Dict = post("https://api.gofile.io/accounts", headers=headers).json()
        if create_account_response["status"] != "ok":
            die("Account creation failed!")
        return create_account_response["data"]["token"]

    def _cacheLink(self, filepath: str, filename: str, link: str) -> None:
        self._files_link_list.append(
            {
                "path": path.join(filepath, filename),
                "filename": filename,
                "link": link
            }
        )

    @staticmethod
    def _downloadContent(file_info: Dict, token: str, chunk_size: int = 4096) -> None:
        if path.exists(file_info["path"]):
            if path.getsize(file_info["path"]) > 0:
                _print(f"{file_info['filename']} already exists, skipping." + NEW_LINE)
                return

        filename: str = file_info["path"] + '.part'
        url: str = file_info["link"]

        headers: Dict = {
            "Cookie": "accountToken=" + token,
            "Accept-Encoding": "gzip, deflate, br",
            "User-Agent": getenv("GF_USERAGENT") if getenv("GF_USERAGENT") else "Mozilla/5.0",
            "Accept": "*/*",
            "Connection": "keep-alive",
            "Pragma": "no-cache",
            "Cache-Control": "no-cache"
        }

        part_size: int = 0
        if path.isfile(filename):
            part_size = int(path.getsize(filename))
            headers["Range"] = f"bytes={part_size}-"

        has_size: str | None = None
        message: str = " "

        try:
            with get(url, headers=headers, stream=True, timeout=(9, 27)) as response_handler:
                if ((response_handler.status_code in (403, 404, 405, 500)) or
                    (part_size == 0 and response_handler.status_code != 200) or
                    (part_size > 0 and response_handler.status_code != 206)):
                    _print(
                        f"Couldn't download the file from {url}."
                        + NEW_LINE
                        + f"Status code: {response_handler.status_code}"
                        + NEW_LINE
                    )
                    return

                has_size = response_handler.headers.get('Content-Length') \
                    if part_size == 0 \
                    else response_handler.headers.get('Content-Range').split("/")[-1]

                if not has_size:
                    _print(
                        f"Couldn't find the file size from {url}."
                        + NEW_LINE
                        + f"Status code: {response_handler.status_code}"
                        + NEW_LINE
                    )
                    return

                with open(filename, 'ab') as handler:
                    total_size: float = float(has_size)
                    start_time: float = perf_counter()
                    for i, chunk in enumerate(response_handler.iter_content(chunk_size=chunk_size)):
                        progress: float = (part_size + (i * len(chunk))) / total_size * 100
                        handler.write(chunk)

                        rate: float = (i * len(chunk)) / (perf_counter()-start_time)
                        unit: str = "B/s"
                        if rate < (1024):
                            unit = "B/s"
                        elif rate < (1024*1024):
                            rate /= 1024
                            unit = "KB/s"
                        elif rate < (1024*1024*1024):
                            rate /= (1024 * 1024)
                            unit = "MB/s"
                        elif rate < (1024*1024*1024*1024):
                            rate /= (1024 * 1024 * 1024)
                            unit = "GB/s"

                        _print("\r" + " " * len(message))
                        message = f"\rDownloading {file_info['filename']}: {part_size + i * len(chunk)}" \
                        f" of {has_size} {round(progress, 1)}% {round(rate, 1)}{unit}"
                        _print(message)
        finally:
            if path.getsize(filename) == int(has_size):
                _print("\r" + " " * len(message))
                message = f"\rDownloading {file_info['filename']}: {path.getsize(filename)} of {has_size} Done!" + NEW_LINE
                _print(message)
                move(filename, file_info["path"])

# Prompt user for input
url = input("Enter the Gofile link (e.g., https://gofile.io/d/z1WJjy): ")
password = input("Enter the password (if any): ")

# Run
_print('Starting, please wait...' + NEW_LINE)
Main(url=url, password=password, max_workers=3)